# VL-JEPA Inference Demo

This notebook demonstrates the three inference modes of VL-JEPA:

1. **Embedding Prediction** — Classification and retrieval without a decoder (2.85x faster)
2. **Visual Question Answering** — Discriminative VQA via embedding matching
3. **Selective Decoding** — Ward clustering to decode only relevant segments

Using smoke-test tiny models. For production, load pretrained checkpoints.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
from src.vljepa.models.vljepa import build_vljepa
from src.vljepa.losses.infonce import BidirectionalInfoNCE

device = torch.device('cpu')  # Use 'cuda' or 'mps' if available
print(f'Device: {device}')

In [ ]:
# Build tiny VL-JEPA for demonstration
config = {
    'x_encoder': {'name': 'vit_tiny', 'embed_dim': 192, 'img_size': 32, 'patch_size': 8, 'depth': 4, 'num_heads': 3},
    'predictor': {'name': 'transformer', 'embed_dim': 192, 'depth': 2, 'num_heads': 3, 'shared_embedding_dim': 192},
    'y_encoder': {'name': 'text_encoder_tiny', 'embed_dim': 192, 'vocab_size': 1000, 'depth': 2, 'num_heads': 3, 'shared_embedding_dim': 192},
}
model = build_vljepa(config).to(device)
model.eval()
print(f'Total params: {model.total_params:,}')
print(f'Trainable params: {model.trainable_params:,}')

## Mode 1: Embedding Prediction (No Decoder)

The primary inference mode. VL-JEPA predicts an embedding directly
in the shared space — no text decoder needed. This is what gives
the 2.85x speedup over autoregressive VLMs.

In [ ]:
# Predict embedding for an image + query
image = torch.randn(1, 3, 32, 32, device=device)
query = torch.randint(0, 1000, (1, 8), device=device)  # "What is this?"

with torch.no_grad():
    embedding = model.forward_embed(image, query)

print(f'Predicted embedding shape: {embedding.shape}')  # (1, 192)
print(f'L2 norm: {embedding.norm().item():.4f}')  # Should be ~1.0

## Mode 2: Zero-Shot Classification via Retrieval

Compare the predicted embedding against class label embeddings.
No retraining needed — just change the labels.

In [ ]:
# Simulate class labels as token sequences
class_labels = ['cat', 'dog', 'car', 'airplane', 'flower']
label_ids = torch.randint(0, 1000, (5, 4), device=device)  # 5 classes, 4 tokens each
label_masks = torch.ones(5, 4, dtype=torch.bool, device=device)

# Encode all labels
with torch.no_grad():
    label_embeddings = model.y_encoder(label_ids, label_masks)  # (5, 192)

# Compare predicted embedding to label embeddings
similarity = embedding @ label_embeddings.T  # (1, 5)
probs = F.softmax(similarity / 0.07, dim=-1)

print('Zero-shot classification results:')
for label, prob in zip(class_labels, probs[0]):
    print(f'  {label}: {prob.item():.2%}')
print(f'\nPredicted: {class_labels[probs.argmax().item()]}')

## Mode 3: Training Demo — Bi-directional InfoNCE

The loss function that makes VL-JEPA work: symmetric contrastive
loss between predicted and target embeddings.

In [ ]:
# Demonstrate InfoNCE loss
loss_fn = BidirectionalInfoNCE(temperature=0.07)

# Simulate a batch of 8 image-text pairs
batch_images = torch.randn(8, 3, 32, 32, device=device)
batch_query = torch.randint(0, 1000, (8, 8), device=device)
batch_target = torch.randint(0, 1000, (8, 16), device=device)
batch_mask = torch.ones(8, 16, dtype=torch.bool, device=device)

# Forward pass
outputs = model.forward_train(
    images=batch_images,
    query_ids=batch_query,
    query_mask=torch.ones(8, 8, dtype=torch.bool, device=device),
    target_ids=batch_target,
    target_mask=batch_mask,
)

# Compute loss
result = loss_fn(outputs['predicted_embedding'], outputs['target_embedding'])
print(f"Loss: {result['loss'].item():.4f}")
print(f"V→T accuracy: {result['accuracy_v2t'].item():.2%}")
print(f"T→V accuracy: {result['accuracy_t2v'].item():.2%}")
print(f"Temperature: {result['temperature'].item():.4f}")
print(f"\nWith random init, accuracy should be ~{1/8:.2%} (1/batch_size).")
print(f"After training, accuracy should approach 100%.")

## Next Steps

1. **Load pretrained models** — Download V-JEPA 2 checkpoint and replace the tiny encoder
2. **Real data** — Use CC3M or WebVid-2M for actual training
3. **Evaluation** — Run on MSR-VTT (retrieval) or GQA (VQA) benchmarks
4. **Selective decoding** — See `selective.py` for the 2.85x speedup
5. **Robotics** — See `04_robotics_planning_demo.ipynb`